In [ ]:
import duckdb 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression ,SGDClassifier
from sklearn.metrics import f1_score


In [ ]:
conn = duckdb.connect(database='catalog.db', read_only=False)
cursor = conn.cursor()

In [ ]:
cursor.execute("""
    SELECT 
        category_id,
        title
    FROM products
    WHERE split='train' AND title IS NOT NULL AND title != '' AND category_id IS NOT NULL
""")
train_data = cursor.fetchall()
print(train_data[:5])  
X_train = [row[1] for row in train_data]
y_train = [row[0] for row in train_data]


In [ ]:
cursor.execute("""
    SELECT 
        category_id,
        title
    FROM products
    WHERE split='test' AND title IS NOT NULL AND title != '' AND category_id IS NOT NULL
""")
test_data = cursor.fetchall()
print(test_data[:5])
X_test = [row[1] for row in test_data]
y_test = [row[0] for row in test_data]


In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
print(f"TF-IDF training data shape: {X_train_tfidf.shape}")
print(f"TF-IDF testing data shape: {X_test_tfidf.shape}")

In [ ]:
clf = SGDClassifier(loss='log_loss', max_iter=1000, random_state=42 ,n_jobs=-1)
clf.fit(X_train_tfidf, y_train)
y_pred = clf.predict(X_test_tfidf)
f1 = f1_score(y_test, y_pred, average='macro')
print(f"F1 Score: {f1}")

In [ ]:
cursor.execute("""
    SELECT
        category_id
    FROM products
    GROUP BY category_id
    HAVING COUNT(*) < 50
""")
small_categories = cursor.fetchall()
print(small_categories)

In [ ]:
import duckdb
from langdetect import detect, LangDetectException
from collections import Counter, defaultdict

conn = duckdb.connect('catalog.db', read_only=True)

sample = conn.execute("""
    SELECT title, category_id
    FROM products 
    WHERE title IS NOT NULL AND title != ''
    ORDER BY random() 
    LIMIT 50000
""").fetchall()
conn.close()

lang_counts = Counter()
category_lang_counts = defaultdict(Counter)
category_totals = Counter()

for title, category_id in sample:
    try:
        lang = detect(title)
    except LangDetectException:
        lang = 'unknown'
    lang_counts[lang] += 1
    category_lang_counts[category_id][lang] += 1
    category_totals[category_id] += 1

total = sum(lang_counts.values())
print("=== التوزيع العام للغات (بعد فلتر /Categories/) ===")
for lang, count in lang_counts.most_common(15):
    print(f"{lang}: {count} ({100*count/total:.2f}%)")

print("\n=== فئات فيها نسبة عالية من العناوين الغير إنجليزية (min 10 عينات في العينة) ===")
risky = []
for category_id, counter in category_lang_counts.items():
    total_cat = category_totals[category_id]
    if total_cat < 10:
        continue
    non_en = total_cat - counter.get('en', 0)
    non_en_ratio = non_en / total_cat
    risky.append((category_id, total_cat, non_en_ratio))

risky.sort(key=lambda x: x[2], reverse=True)
for category_id, total_cat, ratio in risky[:20]:
    print(f"{category_id}: {ratio*100:.1f}% غير إنجليزي (من أصل {total_cat} عينة)")

In [ ]:
import duckdb

conn = duckdb.connect('catalog.db', read_only=True)
result = conn.execute("""
    SELECT COUNT(*) 
    FROM categories c 
    JOIN products p ON c.category_id = p.category_id
    GROUP BY c.category_id, c.category_name
    HAVING COUNT(p.product_id) < 50
""").fetchall()

print("عدد الفئات اللي لسه أقل من 50 منتج (بعد الـ rollup فعلياً):", len(result))
conn.execute("""
SELECT LOWER(TRIM(title)) AS title_key, COUNT(DISTINCT split) AS split_variety
FROM products
GROUP BY LOWER(TRIM(title))
HAVING COUNT(*) > 1
ORDER BY split_variety DESC
LIMIT 10;   
""")
result = conn.fetchall()
for row in result:
    print(f"Title Key: {row[0]}, Split Variety: {row[1]}")
conn.execute("""SELECT COUNT(*) FROM products WHERE category_id = '/Categories';""")
result = conn.fetchone()
print(f"عدد المنتجات اللي لسه في /Categories: {result[0]}")
conn.close()

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

sentence1 = torch.tensor([2, 3, 4])
sentence2 = torch.tensor([5, 6])
sentence3 = torch.tensor([7, 8, 9, 10, 11])

batch = pad_sequence([sentence1, sentence2, sentence3], batch_first=True, padding_value=0)
print(batch)

In [1]:
import torch
import torch.nn as nn

vocab_size = 8482  
embedding_dim = 50 

embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)

sample_input = torch.tensor([2, 3, 4, 0, 0]) 
output = embedding_layer(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([5])
Output shape: torch.Size([5, 50])


In [1]:
import duckdb

conn = duckdb.connect(database='catalog.db', read_only=False)
result = conn.execute("""
    SELECT * FROM read_csv_auto('data/raw/abo-images-small/images/metadata/images.csv.gz')
    LIMIT 5
""").fetchall()
print(result)

[('010-mllS7JL', 106, 106, '14/14fe8812.jpg'), ('01dkn0Gyx0L', 122, 122, 'da/daab0cad.jpg'), ('01sUPg0387L', 111, 111, 'd2/d2daaae9.jpg'), ('1168jc-5r1L', 186, 186, '3a/3a4e88e6.jpg'), ('11RUV5Fs65L', 30, 500, 'd9/d91ab9cf.jpg')]


In [1]:
import duckdb

conn = duckdb.connect(database='catalog.db', read_only=False)

conn.execute("""
    UPDATE product_images
    SET image_path = images_index.full_path
    FROM (
        SELECT 
            image_id, 
            'data/raw/abo-images-small/images/small/' || path AS full_path
        FROM read_csv_auto('data/raw/abo-images-small/images/metadata/images.csv.gz')
    ) AS images_index
    WHERE product_images.image_id = images_index.image_id;
""")

result = conn.execute("SELECT COUNT(*) FROM product_images WHERE image_path LIKE 'data/raw/%';").fetchall()
print("عدد الصور اللي اترّبطت بمسار حقيقي:", result)

result_total = conn.execute("SELECT COUNT(*) FROM product_images;").fetchall()
print("إجمالي عدد صفوف product_images:", result_total)

conn.close()

عدد الصور اللي اترّبطت بمسار حقيقي: [(398170,)]
إجمالي عدد صفوف product_images: [(398170,)]


In [ ]:
import duckdb

conn = duckdb.connect('catalog.db', read_only=True)
result = conn.execute("""
    SELECT products.split, COUNT(*) AS num_images
    FROM product_images
    JOIN products ON products.product_id = product_images.product_id
    WHERE products.split IS NOT NULL
      AND product_images.image_path IS NOT NULL
      AND products.category_id IS NOT NULL
    GROUP BY products.split
""").fetchall()
print(result)

[('train', 190286), ('val', 23205), ('test', 23639)]


: 

In [2]:
from src.dataset.fusion_dataset import load_fusion_data
_, _, train_labels = load_fusion_data('train')
_, _, val_labels = load_fusion_data('val')
train_categories = set(train_labels)
val_categories = set(val_labels)
print('train:', len(train_categories))
print('val:', len(val_categories))
print('missing:', val_categories - train_categories)


train: 219
val: 219
missing: set()


In [1]:
import torch 

torch.cuda.is_available()

False